In [2]:
import cv2
import urllib.request
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
model_path = 'gesture_recognizer.task'
if not os.path.exists(model_path):
    print("Downloading model...")
    url = 'https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'
    urllib.request.urlretrieve(url, model_path)
    print("Download complete.")

In [4]:
action_map = {
    "Closed_Fist": "Shield",
    "Open_Palm": "Damage",
    "Thumb_Up": "Buff",
    "Victory": "Debuff",
    "Pointing_Up": "Magic"
}

# 3. Configure the Recognizer
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(options)

I0000 00:00:1778241246.815787   40926 gl_context.cc:369] GL version: 2.1 (2.1 ATI-7.1.6), renderer: AMD Radeon Pro 5500M OpenGL Engine
W0000 00:00:1778241246.816783   40926 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
I0000 00:00:1778241246.819015   40926 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778241246.853387   41492 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778241246.879719   41497 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778241246.881313   41496 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling 

In [ ]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    # Convert OpenCV frame (BGR) to MediaPipe Image (RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Run the model
    recognition_result = recognizer.recognize(mp_image)
    
    current_action = "None"
    
    # Parse the output
    if recognition_result.gestures:
        top_gesture = recognition_result.gestures[0][0].category_name
        current_action = action_map.get(top_gesture, "None")
        
    # Display the result
    cv2.putText(frame, f"Game Action: {current_action}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Local Gesture Recognition', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1778241248.990875   41498 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


: 

In [ ]:
# Alta strategie, fine tunam model ca sa recunoasca gesturi

In [ ]:
import os
import tensorflow as tf
from mediapipe_model_maker import gesture_recognizer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Verificăm dacă avem imaginile
dataset_path = "dataset"
print(f"Clase detectate: {os.listdir(dataset_path)}")

Clase detectate: ['Zen_inelar', 'Coarne', 'Pumn', 'Ok', 'Pistol', 'Stop', 'Deget_mic', 'Thumbs_down', 'Degete_incrucisate', 'Spiderman', 'Zen', 'Thumbs_up', 'Pinch', 'Shaka', 'Spock', 'None', 'Bagheta', '4', '3', 'L']


In [4]:
# Încărcăm dataset-ul din folder
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

# Împărțim datele: 80% antrenare, 10% validare, 10% testare
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Pumn/Pumn_1_f290_aug1.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Pinch/Pinch_1_f490.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Zen_inelar/Zen_inelar_1_f490.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/None/None_1_f475_aug0.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/L/L_1_f25.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Deget_mic/Deget_mic_1_f460_aug1.jpg


I0000 00:00:1778508192.119445  217606 gl_context.cc:369] GL version: 2.1 (2.1 ATI-7.1.6), renderer: AMD Radeon Pro 5500M OpenGL Engine
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778508192.194403  280556 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778508192.222905  280556 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778508192.262923  280549 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Stop/Stop_1_f150.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Thumbs_up/Thumbs_up_1_f505_aug0.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Thumbs_up/Thumbs_up_2_f55_aug1.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Thumbs_down/Thumbs_down_1_f85_aug0.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Bagheta/Bagheta_1_f580_aug0.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/None/None_1_f435_aug0.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Stop/Stop_1_f285_aug1.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/dataset/Pistol/Pistol_1_f0_aug0.jpg
INFO:tensorflow:Loading image /Users/david/Development/python-ws/hand-battle/

INFO:tensorflow:Load valid hands with size: 11163, num_label: 20, labels: None,3,4,Bagheta,Coarne,Deget_mic,Degete_incrucisate,L,Ok,Pinch,Pistol,Pumn,Shaka,Spiderman,Spock,Stop,Thumbs_down,Thumbs_up,Zen,Zen_inelar.


In [5]:
# Configurația modelului
hparams = gesture_recognizer.HParams(
    learning_rate=0.001,
    export_dir="exported_model",
    epochs=50,             # Poți crește la 100 dacă acuratețea e mică
    batch_size=16
)

options = gesture_recognizer.GestureRecognizerOptions(
    model_options=gesture_recognizer.ModelOptions(dropout_rate=0.05),
    hparams=hparams
)

# Antrenarea propriu-zisă (Transfer Learning pe MobileNetV2)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 20)                2580      
 out (Dense)                                                     
                                                             

INFO:tensorflow:Training the models...


Epoch 1/50
558/558 [==============================] - 3s 5ms/step - loss: 0.6069 - categorical_accuracy: 0.7447 - val_loss: 0.1164 - val_categorical_accuracy: 0.9131 - lr: 0.0010
Epoch 2/50
558/558 [==============================] - 2s 4ms/step - loss: 0.1301 - categorical_accuracy: 0.9037 - val_loss: 0.0663 - val_categorical_accuracy: 0.9668 - lr: 9.9000e-04
Epoch 3/50
558/558 [==============================] - 2s 4ms/step - loss: 0.0899 - categorical_accuracy: 0.9313 - val_loss: 0.0496 - val_categorical_accuracy: 0.9740 - lr: 9.8010e-04
Epoch 4/50
558/558 [==============================] - 2s 4ms/step - loss: 0.0746 - categorical_accuracy: 0.9411 - val_loss: 0.0404 - val_categorical_accuracy: 0.9776 - lr: 9.7030e-04
Epoch 5/50
558/558 [==============================] - 2s 4ms/step - loss: 0.0631 - categorical_accuracy: 0.9467 - val_loss: 0.0375 - val_categorical_accuracy: 0.9767 - lr: 9.6060e-04
Epoch 6/50
558/558 [==============================] - 2s 4ms/step - loss: 0.0553 - catego

In [6]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Acuratețe finală: {acc:.4f}")

1117/1117 [==============================] - 2s 845us/step - loss: 0.0233 - categorical_accuracy: 0.9848
Acuratețe finală: 0.9848


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

print("Generare predicții pentru matricea de confuzie...")
y_true = []
y_pred = []

# Iterăm prin dataset-ul de test
for image, label in test_data.gen_tf_dataset():
    probs = model._model.predict(image, verbose=0)
    
    # Adăugăm rezultatele în liste (batch processing)
    # np.argmax extrage indexul clasei cu probabilitatea cea mai mare
    y_true.extend(np.argmax(label, axis=1))
    y_pred.extend(np.argmax(probs, axis=1))

# Creare matrice folosind listele populate mai sus
cm = confusion_matrix(y_true, y_pred)
labels = test_data.label_names

# Plotting
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='magma')
plt.title('Matricea de Confuzie (Real vs Predicție)')
plt.ylabel('Gestul REAL')
plt.xlabel('Gestul PREZIS de AI')
plt.show()

Generare predicții pentru matricea de confuzie...


/var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/ipykernel_33433/1708074724.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
from sklearn.metrics import classification_report

print("\n--- RAPORT CLASIFICARE PER SEMN ---")
print(classification_report(y_true, y_pred, target_names=labels))


--- RAPORT CLASIFICARE PER SEMN ---
                    precision    recall  f1-score   support

              None       1.00      0.71      0.83         7
                 3       0.98      0.90      0.94        51
                 4       1.00      0.98      0.99        66
           Bagheta       0.98      1.00      0.99        62
            Coarne       1.00      1.00      1.00        51
         Deget_mic       1.00      1.00      1.00        48
Degete_incrucisate       1.00      0.96      0.98        56
                 L       0.98      1.00      0.99        61
                Ok       1.00      1.00      1.00        53
             Pinch       1.00      1.00      1.00        56
            Pistol       0.96      1.00      0.98        51
              Pumn       1.00      1.00      1.00        77
             Shaka       0.97      1.00      0.98        60
         Spiderman       1.00      0.98      0.99        48
             Spock       0.99      0.97      0.98        79
  

In [12]:
# Accesăm istoricul prin modelul intern ._model
history_dict = model._model.history.history

acc = history_dict['accuracy']
val_acc = history_dict['val_accuracy']
loss = history_dict['loss']
val_loss = history_dict['val_loss']

plt.figure(figsize=(14, 5))

# Grafic Acuratețe
plt.subplot(1, 2, 1)
plt.plot(acc, label='Antrenare (Train)')
plt.plot(val_acc, label='Validare (Test)')
plt.title('Evoluția Acurateței')
plt.xlabel('Epoci')
plt.ylabel('Acuratețe')
plt.legend()

# Grafic Pierdere (Loss)
plt.subplot(1, 2, 2)
plt.plot(loss, label='Antrenare (Train)')
plt.plot(val_loss, label='Validare (Test)')
plt.title('Evoluția Pierderii (Loss)')
plt.xlabel('Epoci')
plt.ylabel('Pierdere')
plt.legend()

# Salvare pentru a evita eroarea de backend pe macOS
plt.savefig('evolutie_antrenare.png', dpi=300)
print("Graficele au fost salvate în 'evolutie_antrenare.png'")
plt.show()

KeyError: 'accuracy'

In [15]:
# Căutăm orice atribut care conține 'history'
found_history = None

for attr in dir(model):
    if "hist" in attr.lower():
        candidate = getattr(model, attr)
        # Verificăm dacă e dicționar și are date
        if isinstance(candidate, dict) and len(candidate) > 0:
            found_history = candidate
            print(f"✅ Am găsit istoricul în: model.{attr}")
            break
        # Verificăm dacă e obiectul History de la Keras
        elif hasattr(candidate, 'history') and isinstance(candidate.history, dict) and len(candidate.history) > 0:
            found_history = candidate.history
            print(f"✅ Am găsit istoricul în: model.{attr}.history")
            break

if found_history:
    print("Chei găsite:", found_history.keys())
else:
    print("❌ Istoricul este definitiv gol sau inaccesibil.")

✅ Am găsit istoricul în: model._history.history
Chei găsite: dict_keys(['loss', 'categorical_accuracy', 'val_loss', 'val_categorical_accuracy', 'lr'])


In [16]:
# Extragem datele folosind cheile tale specifice
history_dict = found_history # Sau unde ai stocat rezultatul scanării

acc = history_dict['categorical_accuracy']
val_acc = history_dict['val_categorical_accuracy']
loss = history_dict['loss']
val_loss = history_dict['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

# --- Grafic Acuratețe ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Antrenare (Categorical Acc)')
plt.plot(epochs_range, val_acc, label='Validare (Val Categorical Acc)')
plt.title('Evoluția Acurateței')
plt.xlabel('Epoci')
plt.ylabel('Acuratețe')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# --- Grafic Pierdere (Loss) ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Antrenare (Loss)')
plt.plot(epochs_range, val_loss, label='Validare (Val Loss)')
plt.title('Evoluția Pierderii (Loss)')
plt.xlabel('Epoci')
plt.ylabel('Pierdere')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

# Salvare și afișare
plt.tight_layout()
plt.savefig('performanta_model_final.png', dpi=300)
plt.show()

print("Graficele au fost generate cu succes!")

Graficele au fost generate cu succes!


/var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/ipykernel_33433/476025276.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
model.export_model()

Using existing files at /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/model_maker/gesture_recognizer/gesture_embedder.tflite
Using existing files at /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/model_maker/gesture_recognizer/hand_landmark_full.tflite
Using existing files at /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/model_maker/gesture_recognizer/canned_gesture_classifier.tflite
INFO:tensorflow:Assets written to: /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/tmp8v1so651/saved_model/assets


INFO:tensorflow:Assets written to: /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/tmp8v1so651/saved_model/assets
2026-05-11 17:37:20.305226: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-11 17:37:20.305240: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-11 17:37:20.305438: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/tmp8v1so651/saved_model
2026-05-11 17:37:20.306856: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-11 17:37:20.306866: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/nj/82jpfdhs4n357wtr3qn7dkkh0000gn/T/tmp8v1so651/saved_model
2026-05-11 17:37:20.309301: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-11 17:37:20.338779: I tensorflow/cc/saved_model/l